In [44]:
from langchain_classic.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq




In [45]:

loader=PyPDFLoader('data/Employee-Handbook-Template.pdf')
docs=loader.load()
splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,chunk_overlap=150
)
chunks=splitter.split_documents(docs)
print("Total chunks:", len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)


Total chunks: 26
Employee Handbook Template 
 
Free Employee Handbook Template Including Policies 
and Procedures 
These sample policies and procedures are provided to guide you in developing your 
employee handbook or other policy documentation. Perfect for use in a small business, or an 
enterprise level corporate policy setting.  
Introduction 
Welcome to the Company. This handbook was developed to provide you with an information 
resource for common questions and concerns. If you have questions or concerns about the 
policies outlined here, you should contact your manager or Human Resources. The policies 
stated in this handbook are subject to change at any time at the sole discretion of the 
Company. From time to time, you may receive updated information regarding any changes in
{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2021-01-07T07:17:26+00:00', 'title': 'Employee Handbook Template', 'author': 'hrVillage.com', 'subject': 'Employee Handb

In [46]:
embeddings=GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')

In [47]:
vector_database=Chroma.from_documents(
    documents=docs,embedding=embeddings,collection_name="reranking_rag"
)

In [48]:
results = vector_database.similarity_search(
    query = "On which dates are employees paid?",
    k=5
)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:300])
    print(doc.metadata)


Result 1:
Pay and Compensation 
Employees are paid on the 1st and 15th of each month. 
New employees are required to complete an Employee's Withholding Allowance Certificate 
(W-4). Employees must present their Social Security card to Employment when completing 
the required forms. Optional forms employees ma
{'title': 'Employee Handbook Template', 'author': 'hrVillage.com', 'page_label': '3', 'page': 2, 'moddate': '2021-01-07T07:17:26+00:00', 'producer': 'www.ilovepdf.com', 'source': 'data/Employee-Handbook-Template.pdf', 'creationdate': '2021-01-07T07:17:26+00:00', 'keywords': 'Employee Handbook Template;templates', 'creator': 'Microsoft® Word 2016', 'total_pages': 7, 'subject': 'Employee Handbook Template'}

Result 2:
 Independence Day (4th of July) 
 Labor Day 
 Thanksgiving Day 
 Friday after Thanksgiving Day 
 Christmas Eve 
 Christmas Day 
 New Year's Eve 
Personal Days 
Employees are eligible for 4 paid personal days per calendar year. New employees will 
accrue 1 perso

In [49]:
import os
from dotenv import load_dotenv
import cohere

load_dotenv()

co = cohere.ClientV2(
    api_key=os.getenv("COHERE_API_KEY")
)

print("Cohere client initialized!")

Cohere client initialized!


In [ ]:
query = "On which dates are employees paid?"

# Retrieve candidate chunks from ChromaDB
results = vector_database.similarity_search(
    query,
    k=20
)

# Extract text from the retrieved chunks
documents_to_rerank = [
    doc.page_content for doc in results
]

# Rerank the chunks using Cohere
response = co.rerank(
    model="rerank-v3.5",
    query=query,
    documents=documents_to_rerank,
    top_n=5
)

# Display reranked results
for item in response.results:
    doc = results[item.index]

    print(f"\nRerank score: {item.relevance_score:.4f}")
    print(f"Original metadata: {doc.metadata}")
    print(doc.page_content[:300])
    print("-" * 60)


Rerank score: 0.6623
Original metadata: {'page': 2, 'subject': 'Employee Handbook Template', 'source': 'data/Employee-Handbook-Template.pdf', 'creator': 'Microsoft® Word 2016', 'author': 'hrVillage.com', 'keywords': 'Employee Handbook Template;templates', 'creationdate': '2021-01-07T07:17:26+00:00', 'page_label': '3', 'moddate': '2021-01-07T07:17:26+00:00', 'producer': 'www.ilovepdf.com', 'title': 'Employee Handbook Template', 'total_pages': 7}
Pay and Compensation 
Employees are paid on the 1st and 15th of each month. 
New employees are required to complete an Employee's Withholding Allowance Certificate 
(W-4). Employees must present their Social Security card to Employment when completing 
the required forms. Optional forms employees ma
------------------------------------------------------------

Rerank score: 0.6114
Original metadata: {'creationdate': '2021-01-07T07:17:26+00:00', 'author': 'hrVillage.com', 'creator': 'Microsoft® Word 2016', 'moddate': '2021-01-07T07:17:26+00:00',

: 